# How Popular Music Has Changed Over Time
### Billboard Hot 100 × MusicBrainz — 1958–2024

**Data sources**
| Source | What we get |
|---|---|
| Billboard Hot 100 CSV (Kaggle) | Chart rank, weeks on chart, peak position, 1958–2024 |
| MusicBrainz REST API | Track duration, release year, genre tags, artist country, label |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import (
    load_merged,
    summary_by_decade,
    plot_duration_over_time,
    plot_genre_shifts,
    plot_country_origins,
    plot_country_over_time,
    plot_weeks_on_chart_over_time,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 1. Load the merged dataset

In [ ]:
df = load_merged()   # loads data/processed/merged_dataset.csv
print(f'Rows: {len(df):,}  |  Columns: {len(df.columns)}')
print(f'Date range: {df["chart_date"].min().date()} → {df["chart_date"].max().date()}')
print(f'Unique songs: {df[["title","artist"]].drop_duplicates().shape[0]:,}')
df.head()

In [ ]:
# Data coverage
coverage = pd.DataFrame({
    'Column': df.columns,
    'Non-null %': (df.notna().mean() * 100).round(1).values
}).sort_values('Non-null %', ascending=False)
display(coverage)

## 2. Summary by Decade

In [ ]:
summary = summary_by_decade(df)
display(summary)

## 3. Song Duration Over Time
The 3-minute radio edit became an industry standard — but did streaming change it?

In [ ]:
fig = plot_duration_over_time(df, by='year')
plt.show()

In [ ]:
fig = plot_duration_over_time(df, by='decade')
plt.show()

## 4. Genre Shifts by Decade
Rock → Pop → Hip-Hop — let's see it in the data.

In [ ]:
fig = plot_genre_shifts(df, top_n=8)
plt.show()

## 5. Where Do Artists Come From?
Has the Hot 100 become more or less American over time?

In [ ]:
fig = plot_country_origins(df, top_n=10)
plt.show()

In [ ]:
fig = plot_country_over_time(df, top_countries=5)
plt.show()

## 6. Chart Longevity — Are Hits Sticking Around Longer?
Streaming has changed how songs accumulate chart weeks.

In [ ]:
fig = plot_weeks_on_chart_over_time(df)
plt.show()

## 8. Deep Dive — Your Custom Analysis
Use these cells to explore specific questions.

In [ ]:
# Example: duration distribution by decade (box plot)
sub = df.dropna(subset=['duration_sec', 'decade'])
decades = [d for d in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']
           if d in sub['decade'].unique()]

fig, ax = plt.subplots(figsize=(12, 5))
data_per_decade = [sub[sub['decade'] == d]['duration_sec'].dropna() for d in decades]
ax.boxplot(data_per_decade, labels=decades, patch_artist=True)
ax.set_title('Song Duration Distribution by Decade')
ax.set_ylabel('Duration (seconds)')
ax.set_xlabel('Decade')
ax.axhline(210, color='red', linestyle='--', linewidth=0.8, label='3:30 min')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Example: top labels over time
import re

sub = df.dropna(subset=['mb_label', 'decade']).copy()
# Normalize common label names
sub['mb_label'] = sub['mb_label'].str.strip()
top_labels = sub['mb_label'].value_counts().head(6).index.tolist()
sub_top = sub[sub['mb_label'].isin(top_labels)]

pivot = (
    sub_top.groupby(['decade', 'mb_label'])
    .size()
    .unstack(fill_value=0)
)
pivot_pct = pivot.div(sub.groupby('decade').size(), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 5))
for label in pivot_pct.columns:
    ax.plot(pivot_pct.index, pivot_pct[label], marker='o', label=label, linewidth=2)
ax.set_title('Top Record Labels — Share of Hot 100 Entries by Decade')
ax.set_ylabel('Share (%)')
ax.set_xlabel('Decade')
ax.legend(bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()